In [265]:
import csv
import pandas # this is used to read the dataset
import time # this will be used to track the training time of the model
import torch # this will be the underlying framework of the model
import torch.amp # this will be used for faster processing on the gpu
from torch_geometric.nn import GCNConv, global_mean_pool # this will be used to create 
# the architecture of the model
from torch_geometric.data import Data, Batch, collate as collate_fn # Data acts like the training data while Batch allows for a higher amount of data
# to be passed to the model. Collate is what allows the batches of data to be properly fed to the model
from torch.utils.data import DataLoader, Dataset # this allows the creation and loading
# of the dataset
import rdkit.Chem # this will be used to convert the molecular formula (in the form of
# a SMILES) into a molecule for further processing

In [266]:
if torch.cuda.is_available():
    train_on = 'cuda' # this will use the gpu if possible
    scalar = torch.amp.GradScaler() # this is used to target AI cores on the gpu
else:
    train_on = 'cpu' # if the gpu isn't available, use the cpu

In [267]:
csv_path='tox21.csv'
try:
    read_csv = pandas.read_csv(csv_path, na_values=['NA', 'NaN']) # read the dataset and replace NaN
    # values with NA
except Exception: # if an error occurred... 
    print(f"'{csv_path}' could not be read!") # tell the user that the file could not be read

In [268]:
h_receptors = ['NR-AR', 'NR-AR-LBD', 'NR-AhR',
               'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
               'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
               'SR-HSE', 'SR-MMP', 'SR-p53'] # this defines the 12 human receptors which the model will have to predict.

In [269]:

class Tox21Dataset(Dataset): # this creates a child class which inherits from the Dataset
    # class
    def __init__(self, dataframe): # this calls the constructor (the initializer of the class) with a parameter of dataframe
        super().__init__() # this allows for commands to be sent to the parent class (i.e, Dataset)
        self.dataframe = dataframe # a dataframe is similar to how an excel spreadsheet is

    def __len__(self): # this is one of the functions which is expected by the Dataloader.
        return len(self.dataframe) # this returns the length of the dataframe, which will be used by the Dataloader below for data loading

    def __getitem__(self, index): # this gets an item at a given index
        row = self.dataframe.iloc[index] # locate the row at the given parameter (index)
        smiles = str(row['smiles']) # locate the row of smiles and assign it to a variable named smiles

        mol = rdkit.Chem.MolFromSmiles(smiles) # mol stands for molecule. Convert the SMILES (Simplified Molecular Input Line Entry System, a representation of a molecular 
        # formula) to molecule which can then be used by the program for training.

        if mol is None or mol.GetNumAtoms() == 0: # if a None value is given...
            return None # skip it


        number_of_atoms = mol.GetNumAtoms() # this gets the number of atoms in the molecule
        input = torch.zeros(number_of_atoms, 118, dtype=torch.float) # this creates a 118x118 matrix of 0. When an atom/element is accessed,
        # it fills a specific row with a value, which represents that specific atom. It does the same for the other atoms in the list, also
        # ensuring that each new element is given a separate column to properly represent all 118 elements in the periodic table.
        
        for i, atom in enumerate(mol.GetAtoms()): # enumerate assigns a an index value for each atom in the molecule
            atomic_number = atom.GetAtomicNum() - 1 # subtract one to preform 0-indexing (counting from 0 instead of 1)
            if 0 <= atomic_number < 118: # if the atomic number of an atom is in the given range ...
                input[i, atomic_number] = 1.0 # allocate the atom to a row in the matrix

        molecule_index_list = []  # this creates an empty list which will store the positions of the molecule
        for molecule in mol.GetBonds(): # this is used to find the structure of the molecule
            beginning_of_molecule = molecule.GetBeginAtomIdx() # gets the beginning of the molecule
            ending_of_molecule = molecule.GetEndAtomIdx() # gets the end
            if isinstance(beginning_of_molecule, int) and isinstance(ending_of_molecule, int): # if there is an actual value for the beginning and ending
                # of the molecule...
                molecule_index_list.append([ending_of_molecule, beginning_of_molecule]) # append that information to the list of molecule indexes
                molecule_index_list.append([beginning_of_molecule, ending_of_molecule]) # make it also work bidirectionally

        if len(molecule_index_list) == 0: # if the length of the molecule is 0...
            return None # return None

        molecule = torch.tensor(molecule_index_list, dtype=torch.long).t() # this creates a matrix which represents the molecule while
        # switching the x and y axis in a way where the dataloader would prefer

        if molecule.shape[0] != 2: # if the shape of the first dimension of the molecule is not equal to (2, number of edges)...
            return None # return None

        labels = [row.get(receptor) for receptor in h_receptors] # get the receptors from the available receptors and create a list object to store it in

        if any(label not in (0, 1) for label in labels): # if any value is unknown in the dataset...
            return None # skip them

        true_labels = torch.tensor(labels, dtype=torch.float) # this creates a matrix which represents the actual receptors
        # a molecule may/may not bind to.

        data = Data(x=input, edge_index=molecule, y=true_labels) # this sets up the parameters for the data, which will later be used when creating
        # the architecture and training of the model. 
        data.to(train_on) # move data processes to the GPU (if possible) 
        return data # after everything has been assigned, return whatever the data of the molecule

In [270]:


dataset = Tox21Dataset(read_csv) # read the original dataset
dataset = [data for data in dataset if data is not None] # this filters any information in the data if it contains None (or NaN/Not a Number) values

def collate(batch): # this is used to group the bunches the batches together to feed the model. A batch is like a group of items which will be fed to the model
    return Batch.from_data_list(batch) # this creates a batch object from the data list

loader = DataLoader(dataset, batch_size=8192, shuffle=True, collate_fn=collate) # this loads the dataset with 8192 batches of shuffled molecules to train at a time 

[18:38:08] WARNING: not removing hydrogen atom without neighbors
[18:38:09] Explicit valence for atom # 8 Al, 6, is greater than permitted
[18:38:10] Explicit valence for atom # 3 Al, 6, is greater than permitted
[18:38:10] Explicit valence for atom # 4 Al, 6, is greater than permitted
[18:38:11] Explicit valence for atom # 4 Al, 6, is greater than permitted
[18:38:11] Explicit valence for atom # 9 Al, 6, is greater than permitted
[18:38:11] Explicit valence for atom # 5 Al, 6, is greater than permitted
[18:38:12] Explicit valence for atom # 16 Al, 6, is greater than permitted
[18:38:12] Explicit valence for atom # 20 Al, 6, is greater than permitted


In [271]:
class Model(torch.nn.Module):
    def __init__(self, input_dimension, hidden_dimension, h_receptors): # this sets the model's parameters
        super().__init__() # send any commands to the parent class (torch.nn.Module) if necessary
        self.conv = GCNConv(input_dimension, hidden_dimension) # this will act as the backbone which will represent the molecular structure of
        # a given atom using the input_dimension as the input and outputting the output to the hidden dimension
        self.heads = torch.nn.ModuleList([torch.nn.Linear(hidden_dimension, 1) for _ in range(h_receptors)]) # for every receptor of a molecule, 
        # make one prediction.

    def forward(self, data):
        input = data.x # equates a variable, input, with the inputs from the data (as defined two code block above)
        molecule = data.edge_index # does the same thing except it used edges from data
        input = self.conv(input, molecule) # this combines the inputs (the molecular characteristics and edges) to create a graph 

        input = global_mean_pool(input, data.batch) # as it is, the input is the graphs/representations of the molecules. Using global_mean_pool
        # helps to "stack" these graphs together without losing their specific structures and characteristics
        logits = torch.cat([head(input) for head in self.heads], dim=1).squeeze(-1) # concatenate each of the model's predictions while returning a 
        # a scalar value (a tensor/matrix with 0 dimensions, i.e. a matrix with only one value)
        return logits # this returns whatever the value of logits is

In [272]:
model = Model(input_dimension=118, hidden_dimension=64, h_receptors=len(h_receptors)) # this sets the parameters of the model
model.to(train_on) # this moves the model to the gpu for training

criterion = torch.nn.BCEWithLogitsLoss() # this is set up to measure the model's accuracy (or rather, lack of it)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001) # this is set up to adjust the model's settings to make better predictions

training_loops = 500
for loop in range(training_loops): # this loops for training_loops times to train the model.
    model.train() # this sets the model to training mode
    training_start_time = time.time() # this starts tracking the time of the training
    total_loss = 0.0 # this is used to find the total loss when training
    for batch in loader: # this is used to loop through the batches in the dataloader
        optimizer.zero_grad() # reset the gradients to begin a new round of training (like clearing up any old information before starting something new)
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            output = model(batch) # this is the model's prediction
            target_batch_size = batch.y.view(-1, len(h_receptors)) # this tells the code to infer the size of the
            # batch based on the number of human receptors there are.
            loss = criterion(output, target_batch_size) # calculate how far off the model's prediction was from the
            # actual values
            if torch.isnan(loss): # if the loss is not a number (NaN)...
                continue # skip it
            
        scalar.scale(loss).backward() # this tells the model to look back and find what it did wrong
        scalar.step(optimizer) # this tells the model to change its settings (weights and biases) to make better predictions next time
        scalar.update() # this saves the model's new settings
        total_loss += loss.item() # this is used to find the total loss of the current batch
        training_end_time = time.time() # this is used to find out how long the training took.
        print(f"Total loss: {total_loss}, Batch: {batch}, Time: {training_end_time-training_start_time:.4f}") # show the total loss at a given batch and how long it 
        # took

torch.save(model.state_dict(), 'Tox21_model.pth') # save the model's weights/current settings
print("Training Complete!") # affirm that the training is complete



Total loss: 0.6957715153694153, Batch: DataBatch(x=[44697, 118], edge_index=[2, 90212], y=[36780], batch=[44697], ptr=[3066]), Time: 0.2339
Total loss: 0.695420503616333, Batch: DataBatch(x=[44697, 118], edge_index=[2, 90212], y=[36780], batch=[44697], ptr=[3066]), Time: 0.2299
Total loss: 0.695071816444397, Batch: DataBatch(x=[44697, 118], edge_index=[2, 90212], y=[36780], batch=[44697], ptr=[3066]), Time: 0.2265
Total loss: 0.6947132349014282, Batch: DataBatch(x=[44697, 118], edge_index=[2, 90212], y=[36780], batch=[44697], ptr=[3066]), Time: 0.2331
Total loss: 0.6943721771240234, Batch: DataBatch(x=[44697, 118], edge_index=[2, 90212], y=[36780], batch=[44697], ptr=[3066]), Time: 0.2368
Total loss: 0.694017231464386, Batch: DataBatch(x=[44697, 118], edge_index=[2, 90212], y=[36780], batch=[44697], ptr=[3066]), Time: 0.2353
Total loss: 0.6936674118041992, Batch: DataBatch(x=[44697, 118], edge_index=[2, 90212], y=[36780], batch=[44697], ptr=[3066]), Time: 0.2204
Total loss: 0.693317830

In [273]:
# Inference (asking a model for their prediction)
state = torch.load('Tox21_model.pth', map_location=train_on, weights_only=False) # load the model
model.load_state_dict(state) # load the model's weights
model.eval() # set the model to training mode
model.to(train_on) # train on the GPU (if possible)

Model(
  (conv): GCNConv(118, 64)
  (heads): ModuleList(
    (0-11): 12 x Linear(in_features=64, out_features=1, bias=True)
  )
)

In [274]:
# Model inference  (asking the model to make predictions based on user input)

def predict_smiles(smiles, model, device): # this creates a function with three parameters and is similar to the dataset class created above
    mol = rdkit.Chem.MolFromSmiles(smiles) # get the molecule from the given SMILES
    
    if mol is None or mol.GetNumAtoms() == 0: # if a None value is given...
        return None # skip it
    
    number_of_atoms = mol.GetNumAtoms() # this gets the number of atoms in the molecule
    input = torch.zeros(number_of_atoms, 118, dtype=torch.float) # this creates a 118x118 matrix of 0. When an atom/element is accessed,
    # it fills a specific row with a value, which represents that specific atom. It does the same for the other atoms in the list, also
    # ensuring that each new element is given a separate column to properly represent all 118 elements in the periodic table.
            
    for i, atom in enumerate(mol.GetAtoms()): # enumerate assigns a an index value for each atom in the molecule
        atomic_number = atom.GetAtomicNum() - 1 # subtract one to preform 0-indexing (counting from 0 instead of 1)
        if 0 <= atomic_number < 118: # if the atomic number of an atom is in the given range ...
            input[i, atomic_number] = 1.0 # allocate the atom to a row in the matrix
    
    molecule_index_list = []
    for molecule in mol.GetBonds():
        beginning_of_molecule = molecule.GetBeginAtomIdx() # gets the beginning of the molecule
        ending_of_molecule = molecule.GetEndAtomIdx() # gets the end
        if isinstance(beginning_of_molecule, int) and isinstance(ending_of_molecule, int): # if there is an actual value for the beginning and ending
            # of the molecule...
            molecule_index_list.append([ending_of_molecule, beginning_of_molecule]) # append that information to the list of molecule indexes
            molecule_index_list.append([beginning_of_molecule, ending_of_molecule]) # make it bidirectional
    
    if len(molecule_index_list) == 0: # if the length of the molecule is 0...
        return None # return None
    
    molecule = torch.tensor(molecule_index_list, dtype=torch.long).t()#.contiguous() # this creates a matrix which represents the molecule while
    # switching the x and y axis in a way where the dataloader would prefer and makes it contiguous in memory (gives them a separate, continuous block
    # in a computer's memory)
    
    if molecule.shape[0] != 2: # if the shape of the first dimension of the molecule is not equal to (2, number of edges)...
        return None # return None
    
    
    data = Data(x=input, edge_index=molecule, batch=torch.zeros(len(input)), dtype=torch.long) # sets the data's parameters 
    data = data.to(device) # this moves data processes ideally to the GPU
    data.batch = torch.zeros(number_of_atoms, dtype=torch.long, device=device) # this defines data.batch as a matrix of zeros in the shape of 
    # number_of_atoms X number_of_atoms
    with torch.no_grad():
        logits = model(data).squeeze() # this removes all the data dimensions which have a size of 1 and assigns it to a variable name logits.
        # a logit is "the vector you get as output of the last layer of your neural network." source: https://markelic.de/what-is-a-logit/, accessed 8/14/26

        probability = torch.sigmoid(logits) # this is used to calculate probability by using sigmoid activation function.
        # Sigmoid is a mathematical function which is defined in th equation: σ(x) = 1/(1+e^-x) where σ is the name of the function (sigma) and 
        # e is euler's number.

    predictions = {} # create an empty dictionary of key : value pairs to store each prediction
    for i, receptors in enumerate(h_receptors): # for every receptor in the possible receptor list, given them an index and...
        if probability.numel() != len(h_receptors): # check if the probability has the model's predictions for each receptor.
            # if it doesn't, return this message
            return f'''Error: The number of elements in probability does not equal the number of elements
                       of available receptors.'''
        prob = float(probability[i].item()) # get everything from the i (which stands for the current iteration) dimension of probability and convert it into a
        # floating-point number
        logit = float(logits[i].item()) # get everything from logits 
        predictions[receptors] = { # for the current receptor, make a prediction which has the following:
            'probability': float(prob), # if called, return the float conversion of the probability value from prediction_probability
            'receptor': str(receptors), # if called, return the string (text) of receptor
            'predicted class': 1 if prob >= 0.5 else 0, # if the probability is high enough, make the prediction that the
            # molecule binds to a given receptor
            'logit': logit # if called, return the value of logit
        }
    return predictions # return the model's predictions

In [ ]:
smiles = input("Enter a SMILES: ") # ask the user for an input
results = predict_smiles(smiles, model, train_on) # provide the SMILES, model, and training device to the predict_smiles function
print("Prediction for:", smiles)
for receptor, data in results.items(): # gets the model's prediction for each receptor in the data
    print(f"Receptor: {data['receptor']}, Probability: {data['probability']} ({'Active' if data['predicted class'] == 1 else 'Inactive'})")



Prediction for: CCO
Receptor: NR-AR, Probability: 0.29067009687423706 (Inactive)
Receptor: NR-AR-LBD, Probability: 0.3370041251182556 (Inactive)
Receptor: NR-AhR, Probability: 0.36052167415618896 (Inactive)
Receptor: NR-Aromatase, Probability: 0.29712650179862976 (Inactive)
Receptor: NR-ER, Probability: 0.3007007837295532 (Inactive)
Receptor: NR-ER-LBD, Probability: 0.32186195254325867 (Inactive)
Receptor: NR-PPAR-gamma, Probability: 0.346240371465683 (Inactive)
Receptor: SR-ARE, Probability: 0.3368954658508301 (Inactive)
Receptor: SR-ATAD5, Probability: 0.31334739923477173 (Inactive)
Receptor: SR-HSE, Probability: 0.39316096901893616 (Inactive)
Receptor: SR-MMP, Probability: 0.33131709694862366 (Inactive)
Receptor: SR-p53, Probability: 0.3235464096069336 (Inactive)


The results above are for the SMILES CCO, which corresponds to ethanol. It's toxicity, as according to the training file, is:
0,0,0,0,0,0,0,0,0,0,0,0,TOX584,CCO
Where each 0 represents a lack of toxicity to a given receptor.
As seen above, the model has correctly predicted the toxicity of ethanol,